# Lumpy 02: Forecasting

Notebook version: 4

This is the stripped-down replacement for the old full lumpy workbook.

Kept in the default path:

- Zero Forecast as the control
- Legacy SBA/Croston as the continuity baseline
- Tuned SBA/Croston, which selects the SBA smoothing alpha inside the training data
- Seasonal SBA/Croston, which keeps SBA SKU rates but lets the forecast vary by month-of-year
- Boosted SBA Hybrid, which uses XGBoost when installed, forecasts the monthly total shape, then allocates it back to SKUs using tuned SBA rates
- Recent Mean 6m as the cheap demand-level challenger
- TSB as a cheap intermittent-demand comparator
- Phase-1 summaries for monthly totals, 3-month rolling WMAPE, and SKU-horizon allocation

Paused unless explicitly switched on:

- Hurdle Random Forest, because the current version is slow and underperforms the baselines
- Legacy Aggregate Allocation, because the current version over-allocates and should be replaced by the Phase 2 allocation work
- 6-month and 8-month lag sweeps, because the required 3-month lag is currently best on the SKU-horizon check

The archived full notebook is still available as `archive_lumpy_collision_spare_parts_forecasting_full.ipynb`.


In [12]:
import time
KERNEL_START_TIME = time.perf_counter()
print("OK: kernel is running")


OK: kernel is running


In [13]:
from pathlib import Path
import importlib
import sys
import time

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [
        start,
        start.parent,
        start.parent.parent,
        start / "Inchscape Repo",
        start.parent / "Inchscape Repo",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find repo root. Open this notebook from the Inchscape Repo folder "
        "or update find_repo_root() with the project path."
    )


root = find_repo_root()
src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

_lumpy = importlib.import_module("lumpy_forecasting")
_lumpy = importlib.reload(_lumpy)

DEFAULT_MODEL_NAMES = _lumpy.DEFAULT_MODEL_NAMES
LumpyConfig = _lumpy.LumpyConfig
best_model_by_window = _lumpy.best_model_by_window
build_lumpy_feature_inventory = _lumpy.build_lumpy_feature_inventory
build_lumpy_model_frame = _lumpy.build_lumpy_model_frame
build_lumpy_phase1_tables = _lumpy.build_lumpy_phase1_tables
load_lumpy_inputs = _lumpy.load_lumpy_inputs
make_backtest_splits_for_lags = _lumpy.make_backtest_splits_for_lags
run_lumpy_backtest = _lumpy.run_lumpy_backtest
summarize_by_model = _lumpy.summarize_by_model
summarize_monthly_totals = _lumpy.summarize_monthly_totals
write_lumpy_outputs = _lumpy.write_lumpy_outputs

print("OK: lumpy_forecasting imported")
print(f"Project root: {root}")
print(f"Module path: {Path(_lumpy.__file__).resolve()}")


OK: lumpy_forecasting imported
Project root: D:\Lumpy_Fellas-\Inchscape Repo
Module path: D:\Lumpy_Fellas-\Inchscape Repo\src\lumpy_forecasting.py


In [14]:
# Start in fast-check mode so VS Code proves the notebook is working quickly.
# Required benchmark: 18-month forecast window with a 3-month operational lag.
# Full mode compares 9/12/18-month windows, but keeps the required 3-month lag by default.
FAST_RUN = False
SAVE_OUTPUTS = not FAST_RUN

# Phase 2/diagnostic switches. Leave these off for the first pass; they add time.
# The default model set already includes tuned SBA, seasonal SBA, and the boosted SBA hybrid.
RUN_DIAGNOSTIC_LAGS = False
RUN_LEGACY_AGGREGATE = False
RUN_RANDOM_FOREST = False

FORECAST_WINDOWS = (18,) if FAST_RUN else (9, 12, 18)
LAG_OPTIONS = (3, 6, 8) if RUN_DIAGNOSTIC_LAGS else (3,)
REQUIRED_WINDOW_MONTHS = 18
REQUIRED_LAG_MONTHS = 3

config = LumpyConfig(
    train_months=48,
    gap_months=REQUIRED_LAG_MONTHS,
    test_months=REQUIRED_WINDOW_MONTHS,
    step_months=3,
    min_train_months=18,
    max_folds=1 if FAST_RUN else None,
    external_mode="selected",
)

model_names = DEFAULT_MODEL_NAMES
if RUN_LEGACY_AGGREGATE:
    model_names = model_names + ("aggregate_allocation",)
if RUN_RANDOM_FOREST:
    model_names = model_names + ("hurdle_random_forest",)

print("Mode:", "FAST CHECK" if FAST_RUN else "FULL BACKTEST")
print("Models:", model_names)
print("SBA fixes included:", "sba_croston_tuned" in model_names and "seasonal_sba_croston" in model_names)
print("Boosted SBA hybrid included:", "boosted_sba_hybrid" in model_names)
print("Forecast windows:", FORECAST_WINDOWS)
print("Lag options:", LAG_OPTIONS)
print("Required benchmark:", f"{REQUIRED_WINDOW_MONTHS}m window / {REQUIRED_LAG_MONTHS}m lag")
print("Diagnostic lag sweep:", RUN_DIAGNOSTIC_LAGS)
print("Legacy aggregate allocation:", RUN_LEGACY_AGGREGATE)
print("Hurdle random forest:", RUN_RANDOM_FOREST)
print("Max folds per window/lag:", config.max_folds)
print("Save outputs:", SAVE_OUTPUTS)


Mode: FULL BACKTEST
Models: ('zero', 'sba_croston', 'sba_croston_tuned', 'seasonal_sba_croston', 'recent_mean_6', 'tsb', 'boosted_sba_hybrid')
SBA fixes included: True
Boosted SBA hybrid included: True
Forecast windows: (9, 12, 18)
Lag options: (3,)
Required benchmark: 18m window / 3m lag
Diagnostic lag sweep: False
Legacy aggregate allocation: False
Hurdle random forest: False
Max folds per window/lag: None
Save outputs: True


In [15]:
start_time = time.perf_counter()
print("Loading lumpy inputs and building model frame...")

sales, external = load_lumpy_inputs(root, config)
model_data, external_inventory = build_lumpy_model_frame(sales, external, config)
splits = make_backtest_splits_for_lags(
    model_data,
    config,
    gap_month_options=LAG_OPTIONS,
    test_month_options=FORECAST_WINDOWS,
)
feature_inventory = build_lumpy_feature_inventory(model_data, splits, config)

split_summary = (
    splits.groupby(["test_months", "gap_months"], as_index=False)
    .agg(
        folds=("fold_id", "count"),
        first_test_start=("test_start", "min"),
        last_test_end=("test_end", "max"),
        min_train_months=("train_months", "min"),
    )
    .sort_values(["test_months", "gap_months"])
)
required_benchmark_folds = splits.loc[
    splits["test_months"].eq(REQUIRED_WINDOW_MONTHS)
    & splits["gap_months"].eq(REQUIRED_LAG_MONTHS)
]

elapsed = time.perf_counter() - start_time
print(f"OK: data ready in {elapsed:.1f}s")
print("Sales rows:", len(sales))
print("Model rows:", len(model_data))
print("SKUs:", model_data["sku_id"].nunique())
print("Folds:", len(splits))
print("Required benchmark folds:", len(required_benchmark_folds))
print("External source/features:", len(external_inventory))
print("Model features:", len(feature_inventory))

print("Window/lag split summary:")
display(split_summary)

print("External handoff usage:")
display(external_inventory["usage"].value_counts().rename_axis("usage").reset_index(name="count"))

print("Model feature families, previewed on the required 18m/3m benchmark split:")
display(feature_inventory["feature_family"].value_counts().rename_axis("feature_family").reset_index(name="count"))

display(external_inventory)
display(feature_inventory)
display(splits)


Loading lumpy inputs and building model frame...


D:\Lumpy_Fellas-\Inchscape Repo\src\lumpy_forecasting.py:205: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  


OK: data ready in 18.0s
Sales rows: 38909
Model rows: 38909
SKUs: 690
Folds: 32
Required benchmark folds: 9
External source/features: 25
Model features: 89
Window/lag split summary:


,test_months,gap_months,folds,first_test_start,last_test_end,min_train_months
0,9,3,12,2022-11-01,2026-04-01,19
1,12,3,11,2022-11-01,2026-04-01,19
2,18,3,9,2022-11-01,2026-04-01,19


External handoff usage:


...

Model feature families, previewed on the required 18m/3m benchmark split:


,feature_family,count
0,internal_history,32
1,external_known_ahead,23
2,categorical,16
3,external_model_lagged,14
4,calendar,4


,source_column,feature,usage
0,lag_1yr_national_annual_collisions_context,external_known__lag_1yr_national_annual_collis...,known_ahead
1,lag_1yr_national_annual_fatalities_context,external_known__lag_1yr_national_annual_fatali...,known_ahead
2,lag_1yr_national_annual_serious_injuries_context,external_known__lag_1yr_national_annual_seriou...,known_ahead
3,lag_1yr_national_annual_moderate_injuries_context,external_known__lag_1yr_national_annual_modera...,known_ahead
4,lag_1yr_national_annual_minor_injuries_context,external_known__lag_1yr_national_annual_minor_...,known_ahead
5,lag_1yr_national_annual_total_injuries_context,external_known__lag_1yr_national_annual_total_...,known_ahead
6,lag_1yr_national_annual_total_victims_context,external_known__lag_1yr_national_annual_total_...,known_ahead
7,lag_1yr_national_annual_motorization_rate_context,external_known__lag_1yr_national_annual_motori...,known_ahead
8,lag_1yr_national_annual_vehicles_per_100_peopl...,external_known__lag_1yr_national_annual_vehicl...,known_ahead
9,lag_1yr_national_annual_vehicle_parc_context,external_known__lag_1yr_national_annual_vehicl...,known_ahead


,feature,feature_family,non_null_share_preview_train
0,month_cos,calendar,1.0
1,month_number,calendar,1.0
2,month_sin,calendar,1.0
3,quarter,calendar,1.0
4,Brand_SUZUKI,categorical,1.0
...,...,...,...
84,sku_positive_rate,internal_history,1.0
85,sku_recent_12m_mean,internal_history,1.0
86,sku_recent_3m_mean,internal_history,1.0
87,sku_recent_6m_mean,internal_history,1.0


,fold_id,train_start,train_end,gap_months,test_start,test_end,train_months,test_months,window_label,is_required_18m_3m_benchmark,global_fold_id
0,12,2021-01-01,2022-07-01,3,2022-11-01,2023-07-01,19,9,9_month_test_lag_3m,False,1
1,11,2021-01-01,2022-10-01,3,2023-02-01,2023-10-01,22,9,9_month_test_lag_3m,False,2
2,10,2021-01-01,2023-01-01,3,2023-05-01,2024-01-01,25,9,9_month_test_lag_3m,False,3
3,9,2021-01-01,2023-04-01,3,2023-08-01,2024-04-01,28,9,9_month_test_lag_3m,False,4
4,8,2021-01-01,2023-07-01,3,2023-11-01,2024-07-01,31,9,9_month_test_lag_3m,False,5
5,7,2021-01-01,2023-10-01,3,2024-02-01,2024-10-01,34,9,9_month_test_lag_3m,False,6
6,6,2021-01-01,2024-01-01,3,2024-05-01,2025-01-01,37,9,9_month_test_lag_3m,False,7
7,5,2021-01-01,2024-04-01,3,2024-08-01,2025-04-01,40,9,9_month_test_lag_3m,False,8
8,4,2021-01-01,2024-07-01,3,2024-11-01,2025-07-01,43,9,9_month_test_lag_3m,False,9
9,3,2021-01-01,2024-10-01,3,2025-02-01,2025-10-01,46,9,9_month_test_lag_3m,False,10


In [16]:
start_time = time.perf_counter()
print(f"Running {len(model_names)} model(s) over {len(splits)} fold(s)...")
print("This cell is the main work. In fast-check mode it should finish quickly.")

forecasts = run_lumpy_backtest(
    data=model_data,
    splits=splits,
    config=config,
    model_names=model_names,
)

phase1_tables = build_lumpy_phase1_tables(
    model_data=model_data,
    splits=splits,
    forecasts=forecasts,
)

model_summary = phase1_tables["model_summary"]
monthly_totals = phase1_tables["monthly_totals"]
best_models = phase1_tables["best_model"]
monthly_total_model_summary = phase1_tables["monthly_total_model_summary"]
sku_horizon_model_summary = phase1_tables["sku_horizon_model_summary"]
metric_suite_model_summary = phase1_tables["metric_suite_model_summary"]
lag_comparison_agent_report = phase1_tables["lag_comparison_agent_report"]

elapsed = time.perf_counter() - start_time
print(f"OK: backtest and phase-1 summaries finished in {elapsed:.1f}s")
print("Forecast rows:", len(phase1_tables["forecasts"]))
print("Phase-1 tables:", len(phase1_tables))

print("SKU-month WMAPE summary:")
display(model_summary)

print("Best SKU-month model by window:")
display(best_models)

print("Monthly-total summary - often the cleaner demand-planning lens:")
display(monthly_total_model_summary.sort_values(["window_label", "monthly_total_wmape_percent", "model"]))

print("SKU-horizon summary - SKU quantity across the forecast window, separate from exact month timing:")
display(sku_horizon_model_summary.sort_values(["window_label", "horizon_sku_wmape_percent", "model"]))

print("Phase-1 combined metric view:")
display(metric_suite_model_summary)

print("Lag comparison agent:")
display(lag_comparison_agent_report)


Running 7 model(s) over 32 fold(s)...
This cell is the main work. In fast-check mode it should finish quickly.
OK: backtest and phase-1 summaries finished in 207.2s
Forecast rows: 1824109
Phase-1 tables: 29
SKU-month WMAPE summary:


,window_label,test_months,gap_months,model,rows,sku_count,actual_total,forecast_total,absolute_error,bias,positive_actual_rows,positive_forecast_rows,wmape_percent,bias_percent
0,12_month_test_lag_3m,12,3,Zero Forecast,85549,690,71108.0,0.000000,71108.000000,-71108.000000,29118,0,100.000000,-100.000000
1,12_month_test_lag_3m,12,3,SBA Croston,85549,690,71108.0,45739.832820,86234.595556,-25368.167180,29118,76068,121.272706,-35.675546
2,12_month_test_lag_3m,12,3,SBA Croston Tuned,85549,690,71108.0,82373.371575,94569.162700,11265.371575,29118,81912,132.993704,15.842622
3,12_month_test_lag_3m,12,3,Seasonal SBA Croston,85549,690,71108.0,82373.371575,94621.678181,11265.371575,29118,81912,133.067557,15.842622
4,12_month_test_lag_3m,12,3,Recent Mean 6m,85549,690,71108.0,92603.800000,98390.666667,21495.800000,29118,66804,138.367929,30.229791
5,12_month_test_lag_3m,12,3,Boosted SBA Hybrid,85549,690,71108.0,103998.000000,107553.369012,32890.000000,29118,81912,151.253543,46.253586
6,12_month_test_lag_3m,12,3,TSB,85549,690,71108.0,114630.264804,113763.238594,43522.264804,29118,81912,159.986554,61.205863
7,18_month_test_lag_3m,18,3,Zero Forecast,105079,690,85628.0,0.000000,85628.000000,-85628.000000,35481,0,100.000000,-100.000000
8,18_month_test_lag_3m,18,3,SBA Croston,105079,690,85628.0,54755.780838,104459.681532,-30872.219162,35481,90900,121.992434,-36.053883
9,18_month_test_lag_3m,18,3,SBA Croston Tuned,105079,690,85628.0,107636.884140,120834.966751,22008.884140,35481,98982,141.116185,25.702906


Best SKU-month model by window:


,window_label,test_months,gap_months,model,rows,sku_count,actual_total,forecast_total,absolute_error,bias,positive_actual_rows,positive_forecast_rows,wmape_percent,bias_percent
0,12_month_test_lag_3m,12,3,Zero Forecast,85549,690,71108.0,0.0,71108.0,-71108.0,29118,0,100.0,-100.0
1,18_month_test_lag_3m,18,3,Zero Forecast,105079,690,85628.0,0.0,85628.0,-85628.0,35481,0,100.0,-100.0
2,9_month_test_lag_3m,9,3,Zero Forecast,69959,690,59004.0,0.0,59004.0,-59004.0,23961,0,100.0,-100.0


Monthly-total summary - often the cleaner demand-planning lens:


...

,window_label,test_months,gap_months,model,sku_count,positive_actual_skus,zero_actual_skus,forecast_positive_skus,false_positive_zero_actual_skus,missed_positive_skus,actual_total,forecast_total,absolute_error,bias,horizon_sku_wmape_percent,median_positive_sku_wmape_percent,positive_skus_below_70_percent,positive_skus_below_100_percent,false_positive_zero_actual_sku_share_percent
2,12_month_test_lag_3m,12,3,SBA Croston,690.0,6558.0,732.0,6339.0,581.0,800.0,71108.0,45739.832820,55912.743811,-25368.167180,78.630736,68.528183,3357.0,4564.0,79.371585
3,12_month_test_lag_3m,12,3,SBA Croston Tuned,690.0,6558.0,732.0,6826.0,722.0,454.0,71108.0,82373.371575,57723.149118,11265.371575,81.176730,63.064802,3546.0,4440.0,98.633880
4,12_month_test_lag_3m,12,3,Seasonal SBA Croston,690.0,6558.0,732.0,6826.0,722.0,454.0,71108.0,82373.371575,57723.149118,11265.371575,81.176730,63.064802,3546.0,4441.0,98.633880
1,12_month_test_lag_3m,12,3,Recent Mean 6m,690.0,6558.0,732.0,5567.0,361.0,1352.0,71108.0,92603.800000,66880.200000,21495.800000,94.054396,100.000000,2694.0,3218.0,49.316940
6,12_month_test_lag_3m,12,3,Zero Forecast,690.0,6558.0,732.0,0.0,0.0,6558.0,71108.0,0.000000,71108.000000,-71108.000000,100.000000,100.000000,0.0,0.0,0.000000
0,12_month_test_lag_3m,12,3,Boosted SBA Hybrid,690.0,6558.0,732.0,6826.0,722.0,454.0,71108.0,103998.000000,71203.000165,32890.000000,100.133600,75.407611,3100.0,3931.0,98.633880
5,12_month_test_lag_3m,12,3,TSB,690.0,6558.0,732.0,6826.0,722.0,454.0,71108.0,114630.264804,80761.871941,43522.264804,113.576351,81.523901,2864.0,3832.0,98.633880
9,18_month_test_lag_3m,18,3,SBA Croston,690.0,5795.0,250.0,5050.0,171.0,916.0,85628.0,54755.780838,64638.179917,-30872.219162,75.487200,68.367347,2979.0,3834.0,68.400000
10,18_month_test_lag_3m,18,3,SBA Croston Tuned,690.0,5795.0,250.0,5499.0,246.0,542.0,85628.0,107636.884140,72816.799827,22008.884140,85.038539,70.702125,2874.0,3640.0,98.400000
11,18_month_test_lag_3m,18,3,Seasonal SBA Croston,690.0,5795.0,250.0,5499.0,246.0,542.0,85628.0,107636.884140,72816.799827,22008.884140,85.038539,70.702125,2875.0,3640.0,98.400000


Phase-1 combined metric view:


,window_label,test_months,gap_months,model,sku_month_wmape_percent,sku_month_actual_total,sku_month_forecast_total,sku_month_absolute_error,monthly_total_wmape_percent,horizon_sku_wmape_percent,median_positive_sku_wmape_percent,phase1_decision_score_percent
0,12_month_test_lag_3m,12,3,SBA Croston,121.272706,71108.0,45739.832820,86234.595556,36.604104,78.630736,68.528183,78.835849
1,12_month_test_lag_3m,12,3,SBA Croston Tuned,132.993704,71108.0,82373.371575,94569.162700,26.848871,81.176730,63.064802,80.339768
2,12_month_test_lag_3m,12,3,Seasonal SBA Croston,133.067557,71108.0,82373.371575,94621.678181,27.330803,81.176730,63.064802,80.525030
3,12_month_test_lag_3m,12,3,Recent Mean 6m,138.367929,71108.0,92603.800000,98390.666667,37.401980,94.054396,100.000000,89.941435
4,12_month_test_lag_3m,12,3,Zero Forecast,100.000000,71108.0,0.000000,71108.000000,100.000000,100.000000,100.000000,100.000000
5,12_month_test_lag_3m,12,3,Boosted SBA Hybrid,151.253543,71108.0,103998.000000,107553.369012,52.048527,100.133600,75.407611,101.145223
6,12_month_test_lag_3m,12,3,TSB,159.986554,71108.0,114630.264804,113763.238594,67.675600,113.576351,81.523901,113.746168
7,18_month_test_lag_3m,18,3,SBA Croston,121.992434,85628.0,54755.780838,104459.681532,36.709286,75.487200,68.367347,78.062973
8,18_month_test_lag_3m,18,3,SBA Croston Tuned,141.116185,85628.0,107636.884140,120834.966751,32.383751,85.038539,70.702125,86.179491
9,18_month_test_lag_3m,18,3,Seasonal SBA Croston,141.144749,85628.0,107636.884140,120859.425619,32.736409,85.038539,70.702125,86.306565


Lag comparison agent:


,agent,test_months,score_column,required_lag_months,required_lag_best_model,required_lag_score_percent,best_lag_months,best_window_label,best_model,best_score_percent,delta_vs_required_lag_percent,recommendation
0,Lag Comparison Agent,9,horizon_sku_wmape_percent,3,SBA Croston Tuned,81.052362,3,9_month_test_lag_3m,SBA Croston Tuned,81.052362,0.0,Keep the required 3-month lag as the lead benc...
1,Lag Comparison Agent,12,horizon_sku_wmape_percent,3,SBA Croston,78.630736,3,12_month_test_lag_3m,SBA Croston,78.630736,0.0,Keep the required 3-month lag as the lead benc...
2,Lag Comparison Agent,18,horizon_sku_wmape_percent,3,SBA Croston,75.487200,3,18_month_test_lag_3m,SBA Croston,75.487200,0.0,Keep the required 3-month lag as the lead benc...


In [17]:
if SAVE_OUTPUTS:
    output_paths = write_lumpy_outputs(
        root=root,
        model_data=model_data,
        splits=splits,
        forecasts=forecasts,
        external_inventory=external_inventory,
        feature_inventory=feature_inventory,
        phase1_tables=phase1_tables,
    )

    print("OK: saved lumpy outputs")
    display({name: str(path) for name, path in output_paths.items()})
else:
    print("FAST_RUN is on, so outputs were not overwritten.")
    print("Set FAST_RUN = False in the config cell, then rerun all cells to save final outputs.")


OK: saved lumpy outputs


{'model_data': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_model_data.csv',
 'splits': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_backtest_splits.csv',
 'forecasts': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_backtest_forecasts.csv',
 'model_summary': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_model_summary.csv',
 'fold_model_summary': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_fold_model_summary.csv',
 'monthly_totals': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_monthly_total_results.csv',
 'monthly_total_fold_summary': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_monthly_total_fold_summary.csv',
 'monthly_total_model_summary': 'D:\\Lumpy_Fellas-\\Inchscape Repo\\results\\lumpy_outputs\\tables\\lumpy_monthly_total_model_summary.csv',
 'sku_horizon_results': 'D:\\Lumpy_Fellas